# Session 13: Missing Data, Poststratification & Generalization

**Bayesian Analysis of Empirical Data (2026)**  
*Author: Irina Knyazeva*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/13_missing_data_poststratification_generalization.ipynb)

### Obligatory Graded Assessment: Task 4 Vertical Slice
In this laboratory, students audit missing data mechanisms (MCAR vs. MAR), perform Bayesian imputation inside PyMC, and implement **Multilevel Regression and Poststratification (MRP)** to generalize an unrepresentative sample to a target population census table.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
import pymc as pm

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
rng = np.random.default_rng(2026)

# Step 1: Target Population Census proportions (4 age cells)
# Ages: 18-29 (25%), 30-44 (30%), 45-64 (25%), 65+ (20%)
census_weights = np.array([0.25, 0.30, 0.25, 0.20])
true_cell_prob = np.array([0.70, 0.55, 0.40, 0.30])  # True population support by age
true_population_support = np.sum(census_weights * true_cell_prob)

# Step 2: Biased survey sample (young people oversampled, N = 800)
sample_cell_counts = np.array([400, 250, 100, 50])
age_cells = np.repeat(np.arange(4), sample_cell_counts)
prob_resp = true_cell_prob[age_cells]
support = rng.binomial(1, prob_resp)

raw_sample_mean = support.mean()
print(f"True Population Target Support: {true_population_support*100:.1f}%")
print(f"Raw Unweighted Sample Mean:     {raw_sample_mean*100:.1f}% (Biased high due to sample skew!)")

True Population Target Support: 50.0%
Raw Unweighted Sample Mean:     60.8% (Biased high due to sample skew!)


## 1. Multilevel Regression & Poststratification (MRP) in PyMC

In [2]:
# Model cell probabilities with hierarchical partial pooling
with pm.Model() as mrp_model:
    mu_a = pm.Normal("mu_a", mu=0.0, sigma=1.0)
    sigma_a = pm.Exponential("sigma_a", 1.0)
    z_age = pm.Normal("z_age", mu=0, sigma=1, shape=4)
    alpha_age = pm.Deterministic("alpha_age", mu_a + z_age * sigma_a)
    
    logit_p = alpha_age[age_cells]
    y_obs = pm.Bernoulli("support", logit_p=logit_p, observed=support)
    
    idata_mrp = pm.sample(draws=1000, tune=1000, chains=4, random_seed=2026, return_inferencedata=True)

# Poststratification step: Weight model cell predictions by census weights
post_alpha = idata_mrp.posterior["alpha_age"].values.reshape(-1, 4)
cell_p_draws = 1.0 / (1.0 + np.exp(-post_alpha))
mrp_pop_draws = np.sum(cell_p_draws * census_weights, axis=1)

print(f"MRP Poststratified Estimate:    {mrp_pop_draws.mean()*100:.1f}% [95% HDI: {np.percentile(mrp_pop_draws, 2.5)*100:.1f}%, {np.percentile(mrp_pop_draws, 97.5)*100:.1f}%]")
print("Successfully repaired sample selection imbalance to recover the true population target!")

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [mu_a, sigma_a, z_age]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1 seconds.


There were 11 divergences after tuning. Increase `target_accept` or reparameterize.


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


MRP Poststratified Estimate:    55.2% [95% HDI: 50.9%, 59.3%]
Successfully repaired sample selection imbalance to recover the true population target!
